## 1、使用init_chat_model函数初始化模型

In [ ]:
# 把 .env中的配置加载到当前python进程的环境变量中
from dotenv import load_dotenv
from openai import base_url

load_dotenv()

### 1-1-1被init_chat_model支持的供应商，创建模型

In [ ]:


# 导入langchain初始化模型的函数
from langchain.chat_models import  init_chat_model

# 调用init_chat_model函数来初始化模型
# 参数model来指定模型名称，langchain会自动根据名字模型名字来设定base_url，并从环境变量中获取api_key
load_dotenv()
# os.getenv("DEEPSEEK_API_KEY")
model1 = init_chat_model(model="deepseek-flash")

In [ ]:
print(type(model))

### 1-1-2 如果是不被init_chat_model支持的模型，需要手动指定base_url和api_key

In [ ]:
import os

ali_api_key = os.getenv("aliyun")
model2 = init_chat_model(
    model='qwen-max',
    model_provider='openai',  # 如果是langchain不支持的模型，需要指定模型提供者，虽然我们用的是阿里，但是阿里兼容openai，所以这里写openai
    base_url='https://maas.qianwenaiapi.com/compatible-mode/v1',
    api_key=ali_api_key,
)


In [ ]:
print(type(model2))

### 1-2 使用model类来初始化模型
### 使用社区版本的model类来访问

首先需要安装langchain社区依赖

uv add langchain_community

再安装阿里云百炼依赖

uv add dashscope

然后就可以使用model类来初始化模型了

In [ ]:
from langchain_community.chat_models import ChatTongyi

model3 = ChatTongyi(model='qwen-max', )


In [ ]:
print(type(model3))

# 2 调用模型

langchain 提供了2个不同的函数来访问模型

- invoke 阻塞式访问

- stream 流式访问

### 2-1 方法1——阻塞式调用：需要等待模型生成全部的结果才会返回，等待时间较长



In [ ]:
# 只传入字符串
response = model.invoke("你好")
print(response)

In [ ]:
# 传入messages数组
response2 = model.invoke([{"role": "user", "content": "你好"}])

In [ ]:
print(response2)

### 2-2 方法2——stream流式调用：模型生成一部分结果就会返回，等待时间较短

In [ ]:
response = model.stream("你好,介绍一下你自己")


In [ ]:
print(type(response))

In [ ]:
for chunk in response:
    print(chunk.content,end='',flush=True)

# 3-在智能体中使用模型

### 3-1 创建智能体
langchain 提供了一个create_agent函数用来快速创建智能体，调用creat_agent时需要指定一个模型，有两种选择

- 使用初始化好的模型对象

- 使用模型名称，让langchain自动初始化模型

In [80]:
from langchain.agents import create_agent

# 1.使用初始化好的model创建agent
agent = create_agent(model=model, )


In [ ]:
# 2.指定好model名称，由langchain自动初始化模型
# 如果不在langchain默认识别的供应商里面，就只能用方法一，先初始化好模型对象，再用初始化好的模型对象来创建智能体
# agent = create_agent(model='deepseek')


### 3-2

智能体调用和模型调用类似，也支持两种方式

- invoke 阻塞式访问
- stream 流式访问

但是需要注意的是，智能体调用时，需要传入一个dict，其中必须包含一个messages字段，也就是消息的列表

In [81]:
# 1.阻塞式调用
response = agent.invoke({"messages": [{"role": "user", "content":"你是谁"}]})


In [ ]:
print(response)

In [82]:
# 2.流式调用
res = agent.stream(
    # 参数1：messages字段，必须传入
    {"messages": [{"role": "user", "content": "你是谁"}]},
    # 参数2：流的模式（智能体中特有的）
    stream_mode="message",
)


In [83]:
print(type(response))

<class 'dict'>


In [85]:
for chunk,metadata in res:
    if chunk.content:
        print(chunk,end='',flush=True)